# RQ2 — Agreement and relationships between files

Reproduces the subsections *Manual annotation and agreement*, *Sample*, and *Relationships between files*:

| Result in the paper | Where |
|---|---|
| κ between the annotators (conflicting, redundant, and mean); disagreement on 7 of 141 labels | section 1 |
| Sample composition: 44 repositories with two files and 3 with three; format combinations | section 2 |
| Table of prevalence of the relationships with 90% Wilson CIs and finite population correction | section 3 |
| Conflict always accompanied by redundancy; relationships by format combination | section 4 |
| Files that refer the agent to another context file | section 5 |

Inputs: `data/rq02/manual_annotation/rq02_labels_annotator_A.csv` and `..._annotator_B.csv`;
in section 5, also the files in `data/collection/context_files/` and the RQ1 classification
(`data/rq01/classifier/rq01_classification_run_1.csv`). The reference labels are those of
annotator A (see `data/rq02/README.md`). Column names follow the annotation spreadsheet:
`conflitantes` = conflicting, `redundantes` = redundant, `complementares` = complementary.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent

ANNOT_A = ROOT / "data/rq02/manual_annotation/rq02_labels_annotator_A.csv"
ANNOT_B = ROOT / "data/rq02/manual_annotation/rq02_labels_annotator_B.csv"
OUT = ROOT / "results/rq02"
OUT.mkdir(parents=True, exist_ok=True)

RELATIONSHIPS = ["complementares", "redundantes", "conflitantes"]
NAME_EN = {"complementares": "complementary", "redundantes": "redundant", "conflitantes": "conflicting"}
N_FRAME = 149       # repositories in the frame from which the sample was drawn
CONF = 0.90         # same level used to size the sample
Z = stats.norm.ppf(1 - (1 - CONF) / 2)


def repo_key(value):
    m = re.search(r"github\.com/([^/\s]+)/([^/\s]+)", str(value))
    return (f"{m.group(1)}/{m.group(2)}" if m else str(value).strip()).lower()


def load(path):
    d = pd.read_csv(path)
    d["key"] = d["repo"].map(repo_key)
    assert d["status"].eq("Finalizado").all() and not d["key"].duplicated().any()
    return d.set_index("key").sort_index()


A, B = load(ANNOT_A), load(ANNOT_B)
assert A.index.equals(B.index)
print(f"Repositories: {len(A)}")

Repositories: 47


## 1. Agreement between the annotators

The κ of the complementary relationship is undefined: both annotators marked it in all 47 repositories.

In [2]:
def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float("nan") if pe == 1 else (po - pe) / (1 - pe)


k = pd.Series({NAME_EN[r]: kappa(A[r], B[r]) for r in RELATIONSHIPS})
k_mean = k[["conflicting", "redundant"]].mean()
disagreements = int((A[RELATIONSHIPS] != B[RELATIONSHIPS]).values.sum())
print(k.round(3).to_string())
print(f"Mean κ (conflicting, redundant): {k_mean:.3f}")
print(f"Divergent labels: {disagreements} of {A[RELATIONSHIPS].size}")

complementary      NaN
redundant        0.762
conflicting      0.746
Mean κ (conflicting, redundant): 0.754
Divergent labels: 7 of 141


## 2. Sample composition

In [3]:
files = A["arquivos_para_analisar"].str.split(";").apply(lambda xs: tuple(sorted(x.strip() for x in xs)))
print("Files per repository:", files.str.len().value_counts().sort_index().to_dict())
combination = files.apply(" + ".join)
combination.value_counts()

Files per repository: {2: 44, 3: 3}


arquivos_para_analisar
AGENTS.md + COPILOT                21
CLAUDE.md + COPILOT                14
AGENTS.md + CLAUDE.md               9
CLAUDE.md + COPILOT + CURSOR        1
AGENTS.md + CLAUDE.md + CURSOR      1
AGENTS.md + CLAUDE.md + COPILOT     1
Name: count, dtype: int64

## 3. Prevalence of the relationships

Wilson interval with finite population correction: the effective sample size is
n·(N − 1)/(N − n). When the prevalence is 100%, the upper bound is 100%. The table shows the
interval with N = 149 (the frame from which the sample was drawn) and with N = 205 (the repositories
that passed the size filter).

In [4]:
def wilson_fpc(k, n, N, z=Z):
    n_eff = n * (N - 1) / (N - n)
    p = k / n
    den = 1 + z ** 2 / n_eff
    center = (p + z ** 2 / (2 * n_eff)) / den
    half = z * np.sqrt(p * (1 - p) / n_eff + z ** 2 / (4 * n_eff ** 2)) / den
    lo, hi = max(0.0, center - half), min(1.0, center + half)
    return (lo, 1.0 if k == n else hi)


table = []
for r in RELATIONSHIPS:
    kk, n = int(A[r].sum()), len(A)
    lo, hi = wilson_fpc(kk, n, N_FRAME)
    lo205, hi205 = wilson_fpc(kk, n, 205)
    table.append({"relationship": NAME_EN[r], "repositories": kk, "pct": 100 * kk / n,
                  "ci90_low": 100 * lo, "ci90_high": 100 * hi,
                  "ci90_low_N205": 100 * lo205, "ci90_high_N205": 100 * hi205})
table = pd.DataFrame(table)
table.to_csv(OUT / "relationship_prevalence.csv", index=False)
table.round(1)

,relationship,repositories,pct,ci90_low,ci90_high,ci90_low_N205,ci90_high_N205
0,complementary,47,100.0,96.2,100.0,95.7,100.0
1,redundant,39,83.0,74.3,89.2,73.7,89.5
2,conflicting,10,21.3,14.3,30.4,14.0,31.0


## 4. Co-occurrence of the relationships and format combinations

In [5]:
conf, red = A["conflitantes"] == 1, A["redundantes"] == 1
print(f"Conflicting repositories that are also redundant: {(conf & red).sum()} of {conf.sum()}")
print(f"Without redundancy: {(~red).sum()}; of these, only complementary: {(~red & ~conf).sum()}")

by_combination = (A.assign(combination=combination).groupby("combination")[["redundantes", "conflitantes"]]
                  .agg(["sum", "size"]).rename(columns=NAME_EN))
by_combination

Conflicting repositories that are also redundant: 10 of 10
Without redundancy: 8; of these, only complementary: 8


redundant      conflicting     
                                      sum size         sum size
combination                                                    
AGENTS.md + CLAUDE.md                   5    9           3    9
AGENTS.md + CLAUDE.md + COPILOT         1    1           0    1
AGENTS.md + CLAUDE.md + CURSOR          1    1           1    1
AGENTS.md + COPILOT                    17   21           3   21
CLAUDE.md + COPILOT                    14   14           3   14
CLAUDE.md + COPILOT + CURSOR            1    1           0    1

## 5. Files that refer the agent to another file

- **explicit reference**: the file names another context file of the same repository;
- **reference only**: the file received only `doc_metadata` in the RQ1 classification.

In [6]:
CONTEXT_FILES = ROOT / "data/collection/context_files"
RQ01 = ROOT / "data/rq01/classifier/rq01_classification_run_1.csv"
NAME = {"COPILOT": "copilot-instructions.md", "CURSOR": ".cursorrules"}
PATTERN = {"AGENTS.md": r"AGENTS\.md", "CLAUDE.md": r"CLAUDE\.md", "copilot-instructions.md": r"copilot-instructions",
           ".cursorrules": r"\.cursorrules", "GEMINI.md": r"GEMINI\.md"}
RQ01_CATEGORIES = ["run_and_build", "project_overview", "doc_metadata", "test_instructions", "project_structure",
                   "system_design", "contribution_guidelines", "code_conventions", "ai_rules",
                   "performance_&_resource_management", "troubleshooting", "security", "deployment_&_infrastructure"]
folders = {p.name.lower(): p for p in CONTEXT_FILES.iterdir() if p.is_dir()}
rq01 = pd.read_csv(RQ01)
rq01["key"] = rq01["repo"].str.lower() + "|" + rq01["arquivo_analisado"].str.lower()
doc_only = set(rq01.loc[(rq01[RQ01_CATEGORIES].fillna(0).astype(int).sum(axis=1) == 1) & (rq01["doc_metadata"] == 1), "key"])


def read(repo, name):
    folder = folders[repo.replace("/", "__")]
    return next(p for p in folder.iterdir() if p.name.lower() == name.lower()).read_text(encoding="utf-8", errors="ignore")


rows = []
for repo, row in A.iterrows():
    labels = [x.strip() for x in row["arquivos_para_analisar"].split(";")]
    names = [NAME.get(x, x) for x in labels]
    citing = [n for n in names if any(re.search(PATTERN[o], read(repo, n)) for o in names if o != n)]
    pure = [n for lbl, n in zip(labels, names) if n in citing and f"{repo}|{lbl.lower()}" in doc_only]
    rows.append({"repo": repo, "redundant": row["redundantes"], "files_that_refer": ";".join(citing),
                 "reference_only": ";".join(pure)})
ref = pd.DataFrame(rows)
ref.to_csv(OUT / "references_between_files.csv", index=False)
for group, name in [(0, "without redundancy"), (1, "with redundancy")]:
    g = ref[ref["redundant"] == group]
    print(f"{name}: {len(g)} repositories; with a file that refers to another: {(g['files_that_refer'] != '').sum()}; "
          f"of these, with the reference only: {(g['reference_only'] != '').sum()}")
ref[ref["redundant"] == 0]

without redundancy: 8 repositories; with a file that refers to another: 7; of these, with the reference only: 3
with redundancy: 39 repositories; with a file that refers to another: 19; of these, with the reference only: 1


,repo,redundant,files_that_refer,reference_only
4,axonframework/axonframework,0,AGENTS.md,AGENTS.md
8,bigbluebutton/bigbluebutton,0,AGENTS.md,
18,gnolang/gno,0,,
29,microsoft/applicationinsights-java,0,AGENTS.md,AGENTS.md
36,remix-run/react-router,0,CLAUDE.md,
39,rsyslog/rsyslog,0,copilot-instructions.md,
40,runatlantis/atlantis,0,copilot-instructions.md,copilot-instructions.md
41,scylladb/scylladb,0,AGENTS.md,
